# citkid.pipeline_v2 — overview and usage

This notebook introduces the current `citkid.pipeline_v2` workflow end-to-end. The pipeline keeps one active state for each stored parameter, embeds YAML and custom step source into the output zarr, and invalidates downstream data when an earlier step is re-run.

---

## Concepts

The framework has four layers:

| Layer | Class / function | Purpose |
|---|---|---|
| **Step definition** | `plStep` | Describes one calibration or analysis step and its function signature |
| **Data + calibration** | `DataSet` | Lazy zarr-backed parameter store plus calibration-path execution |
| **Analysis execution** | `AnalysisRunner` | Runs analysis steps on a `DataSet` and manages downstream invalidation |
| **Interactive review** | `run_iq_analysis`, `run_ts_analysis`, `run_gain_only_analysis`, `run_sweep_fitter` | Interactive review and manual reruns |

**`data_idx`** is the row index, one entry per tone.  
**`nrows`** is the total number of rows.

`pipeline_v2` is intentionally strict: if an earlier step changes, downstream products are invalidated immediately so the dataset cannot expose mixed upstream/downstream state.

---

## 1. Defining custom steps with `plStep`

Write custom calibration and analysis steps in Python files that expose `custom_cal_steps` and `custom_analysis_steps`. The supported `func_type` values are still `'global'`, `'global-res'`, `'per-row'`, and `'vectorized'.`

If a quantity should vary by `data_idx`, define that step as `'per-row'` or `'vectorized'`. Do not use a global step for values that should differ between rows.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline_v2.framework import plStep

def load_global_data():
    """Load data shared across all rows."""
    root = zarr.open('path/to/data.zarr', mode='r')
    fres_all = np.array(root['fres_all'])
    qres_all = np.array(root['qres_all'])
    nrows = int(root['fres'].shape[0])
    return fres_all, qres_all, nrows

def load_global_res_data():
    """Load per-resonator data computed once for all rows."""
    root = zarr.open('path/to/data.zarr', mode='r')
    return np.array(root['fres']), np.array(root['qres']), np.array(root['ares']), np.array(root['res_idxs'])

def load_data_f(data_idx):
    """Load the fine sweep for one row."""
    root = zarr.open('path/to/data.zarr', mode='r')
    ff = np.array(root['f'][data_idx])
    zf = np.array(root['z'][data_idx])
    idx = np.argsort(ff)
    return ff[idx], zf[idx]

custom_cal_steps = [
    plStep('load_global_data', load_global_data, [], ['fres_all', 'qres_all', 'nrows'], 'global'),
    plStep('load_global_res_data', load_global_res_data, [], ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
    plStep('load_data_f', load_data_f, ['data_idx'], ['ff', 'zf'], 'per-row'),
]

def fit_example(ff, zf):
    """Example analysis step."""
    return np.nanmean(ff), np.nanmean(np.abs(zf))

custom_analysis_steps = [
    plStep('fit_example', fit_example, ['ff', 'zf'], ['fr_est', 'amp_est'], 'per-row'),
]

## 2. DataSet — lazy zarr-backed parameter store

`DataSet` exposes stored and calibratable parameters lazily. A parameter is only loaded or produced when you access it. Active parameters live directly at the top level of the zarr store.

## YAML aliases

Built-in calibration aliases:

| alias | meaning |
|---|---|
| `'iq'` | IQ-only calibration template |
| `'ts'` | on-resonance timestream calibration template |
| `'ts_offres'` | off-resonance timestream calibration template |

In [ ]:
from citkid.pipeline_v2.dataset import DataSet

zarr_path = 'path/to/output_v2.zarr'

DS = DataSet(
    zarr_path = zarr_path,
    cal_yaml_path = 'iq',
    custom_cal_steps = custom_cal_steps,
    # custom_path = 'my_custom_cal_steps.py',
    # custom_main_directory_overwrite = r'new/path/to/raw/data',
    zarr_mode = 'a',
)

### 2a. Accessing parameters

```python
DS.nrows
DS.fres[0]
DS.ff[0]
DS.ff[[0, 1, 2]]
```

Global parameters are returned directly. Per-row parameters are exposed as lazy accessors backed by the top-level zarr arrays.

In [ ]:
print('nrows:', DS.nrows)
print('fres[0]:', DS.fres[0])

ff0 = DS.ff[0]
zf0 = DS.zf[0]

print(DS.root.tree())

### 2b. Embedded definitions and path relocation

On first creation, `DataSet` stores the calibration YAML and custom calibration source inside the zarr metadata. After that, you can reopen the dataset with only the zarr path.

If the embedded custom calibration source contains a top-level `main_directory` assignment and you moved the raw data, use `custom_main_directory_overwrite` when reopening to swap in the new root path before the custom steps are compiled.

In [ ]:
# Later, possibly on another machine
DS_reloaded = DataSet(
    zarr_path = zarr_path,
    # custom_main_directory_overwrite = r'new/path/to/raw/data',
)
print(DS_reloaded.nrows)

In [ ]:
# Example: temporarily replace a calibration input
result = DS.apply_cal(
    data_indices = [0, 1, 2],
    outputs = ['ff'],
    replacements = {
        # 'zt': np.array([...]),
    },
)
print(result.keys())

### 2c. Temporary calibration replacements with `apply_cal`

Use `apply_cal(...)` when you want to evaluate some calibration outputs with temporary replacement inputs without mutating dataset memory or saving intermediary results. This is useful for what-if calculations such as computing `xt` from a replacement timestream.

The replacement parameters must match the expected scope: global replacements are scalars, while per-row replacements must provide one value per requested `data_idx`.

## 3. AnalysisRunner — execute the analysis pipeline

`AnalysisRunner` executes a YAML-defined analysis path on top of a `DataSet`. The analysis YAML and custom analysis source are embedded in the zarr output the first time the runner is created.

`pipeline_v2` is strict about dependency safety: re-running an earlier analysis step invalidates all later analysis outputs and any calibration products that depend on them.

In [ ]:
from citkid.pipeline_v2.analysis import AnalysisRunner

AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'iq',
    custom_path = 'my_custom_analysis_steps.py',
)

### 3a. Run the full analysis path

`execute_path(...)` saves by default. Pass `save=False` only when you explicitly want results to remain in memory until a later `save_step_outputs(...)` call.

In [ ]:
AR.execute_path(verbose = True)

### 3a-bis. Controlling execution order and memory usage

There are now two separate knobs:

| Parameter | Meaning |
|---|---|
| `execution_mode='vectorized'` | Run vectorized steps in their normal all-rows-at-once mode |
| `execution_mode='per-row'` | Force vectorized steps to dispatch one row at a time |
| `execute_per_row=True` | Run the entire remaining analysis path row-by-row: row 0 through all steps, then row 1, and so on |

Use `execution_mode='per-row'` when only the vectorized steps are too memory-heavy. Use `execute_per_row=True` when you want the whole path to proceed one row at a time.

In [ ]:
# Lower-memory dispatch for vectorized steps
AR.execute_path(
    execution_mode = 'per-row',
    verbose = True,
    save = True,
    data_idx = np.arange(DS.nrows),
)

# True row-major execution: complete one row through the full path before the next
AR.execute_path(
    data_idx = np.arange(DS.nrows),
    execute_per_row = True,
    verbose = True,
    save = True,
)

### 3b. Re-running a single step safely

`execute_step(...)` also saves by default. If you want to experiment in memory first and decide later whether to persist the result, pass `save=False`.

Re-running an earlier step invalidates later products immediately. `execute_step(...)` only runs the step you asked for; it does not silently rerun earlier analysis steps to satisfy missing analysis inputs.

If a requested step depends on an earlier analysis output that is missing, `execute_step(...)` raises and tells you which earlier analysis step must be run first.

Re-running a `global` or `global-res` analysis step after its outputs already exist requires `allow_global_step_overwrite=True`.

In [ ]:
fit_iq_step = next(s['task'] for s in AR.path if s['task'].name == 'fit_iq')

custom_mask = np.ones(DS.ff[5].shape, dtype=bool)
custom_mask[:10] = False

# save=True is the default
AR.execute_step(
    fit_iq_step,
    data_idx = [5],
    user_params = {'iq_mask': custom_mask},
)

# If you rerun a global/global-res analysis step after outputs already exist,
# you must pass allow_global_step_overwrite = True explicitly.

### 3c. Keep results in memory until you choose to save

Because `execute_step(...)` and `execute_path(...)` save by default, the main reason to call `save_step_outputs(...)` is when you intentionally ran with `save=False` during interactive exploration and now want to persist the current inputs and outputs without re-running the step.

In [ ]:
AR.execute_step(
    fit_iq_step,
    data_idx = [7],
    user_params = {'iq_mask': None},
    save = False,
    # save=False keeps this result in memory only for now
)
AR.save_step_outputs(fit_iq_step, data_idx = [7])

### 3d. Re-opening without extra file paths

Once the analysis definition is embedded, you can re-open both the dataset and runner with just the zarr path.

In [ ]:
DS2 = DataSet(zarr_path = zarr_path)
AR2 = AnalysisRunner(DS2)
print([step_dict['task'].name for step_dict in AR2.path])

## 4. Interactive analysis

Import the interactive entry points from `citkid.pipeline_v2.interactive` and pass in a V2 `AnalysisRunner`.

Important interactive behavior:

- Running one panel invalidates downstream panels for that `data_idx` and clears their plots.
- The UI asks for confirmation before saving or navigating away while downstream panels still need a rerun.
- Global/global-res analysis prefixes are only executed automatically when their outputs do not already exist.

In [ ]:
from citkid.pipeline_v2.interactive import run_iq_analysis, run_ts_analysis

on_res_idxs = np.where(DS.res_idxs[:] >= 0)[0]

run_iq_analysis(
    AR,
    start_idx = 0,
    data_idxs = on_res_idxs,
    title = 'IQ Analysis V2',
    ui_scale = 1.0,
    plot_scale = 1.0,
)

## Summary

Use `citkid.pipeline_v2` when you want a strict single-state pipeline with embedded YAML/custom-step definitions, immediate downstream invalidation, and interactive tools that make stale downstream state explicit rather than silently keeping inconsistent results.